# SLERP-Fixed: Close-Out Eval

The only remaining open item. The geometric-mean-norm-interpolation fix was
already validated on raw deltas (ratio corrected 1.13 -> 0.90) and the fixed
checkpoint (`Qwen3-0.6B-slerp-FIXED-3-adapters-merged-2`) is already confirmed
loadable and working — it was used successfully in the attn/mlp CKA notebook.

No merge step here. This notebook ONLY evaluates the existing checkpoint —
same harness (build_chat_prompt, evaluate_gsm8k, evaluate_humaneval,
evaluate_dolly_perplexity) used for every other method, so the numbers are
directly comparable.

In [4]:
!pip install unsloth_zoo

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 22.6 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 85.5 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 113.9 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 29.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 225.0/225.0 kB 15.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 215.0/215.0 kB 15.8 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency 

In [1]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps "xformers=={xformers}" trl peft accelerate bitsandbytes unsloth_zoo
    !pip install --no-deps unsloth


## HF Token

Same as the ct_calibrated_merge notebook — needed for `push_to_hub`-free authenticated pulls (avoids the anonymous rate-limit warnings you saw last run) and in case the repo needs re-verifying.

In [ ]:
HF_TOKEN = key # Insert your own token

from huggingface_hub import login
login(token=HF_TOKEN)


## Evaluation Harness (identical to ct_calibrated_merge notebook)

In [5]:
import torch
import gc
import re
import math
import multiprocessing
import contextlib
import io
from datasets import load_dataset
from tqdm import tqdm
from unsloth import FastLanguageModel

def build_chat_prompt(tokenizer, user_content: str) -> str:
    messages = [{"role": "user", "content": user_content}]
    return tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True, enable_thinking=False,
    )

def prep_tokenizer_for_generation(tokenizer):
    tokenizer.padding_side = "left"
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    return tokenizer

# ── GSM8K ──
def extract_gsm8k_answer(text: str) -> str:
    hash_match = re.findall(r"####\s*(-?[\d,]+\.?\d*)", text)
    if hash_match:
        return hash_match[-1].replace(",", "").strip()
    boxed = re.findall(r"\\boxed\{([^}]+)\}", text)
    if boxed:
        return boxed[-1].replace(",", "").strip()
    numbers = re.findall(r"-?\d[\d,]*\.?\d*", text)
    if numbers:
        return numbers[-1].replace(",", "").strip()
    return text.strip()

def format_gsm8k_prompt(question: str) -> str:
    return (f"Solve the following math problem. Show your reasoning and put "
            f"your final numeric answer after '#### '.\n\nQuestion: {question}")

def evaluate_gsm8k(model, tokenizer, model_name="model", num_samples=200, batch_size=4,
                    max_new_tokens=320, device="cuda"):
    print(f"\n{'─'*60}\n[GSM8K] Evaluating: {model_name}\n{'─'*60}")
    model.eval()
    prep_tokenizer_for_generation(tokenizer)
    dataset = load_dataset("gsm8k", "main", split="test")
    dataset = dataset.select(range(min(num_samples, len(dataset))))
    preds, labels = [], []
    for i in tqdm(range(0, len(dataset), batch_size), desc=f"{model_name} [gsm8k]"):
        batch = dataset[i : i + batch_size]
        questions    = batch["question"]
        true_answers = [extract_gsm8k_answer(a) for a in batch["answer"]]
        prompts      = [build_chat_prompt(tokenizer, format_gsm8k_prompt(q)) for q in questions]
        inputs = tokenizer(prompts, return_tensors="pt", padding=True, truncation=True,
                            max_length=512, add_special_tokens=False).to(device)
        with torch.no_grad():
            outputs = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False,
                                      pad_token_id=tokenizer.pad_token_id, eos_token_id=tokenizer.eos_token_id,
                                      repetition_penalty=1.3)
        input_len = inputs["input_ids"].shape[1]
        for j, output in enumerate(outputs):
            generated = tokenizer.decode(output[input_len:], skip_special_tokens=True)
            preds.append(extract_gsm8k_answer(generated))
            labels.append(true_answers[j])
    per_sample_exact = [int(p.strip() == l.strip()) for p, l in zip(preds, labels)]
    exact_match = round(sum(per_sample_exact) / len(per_sample_exact), 4)
    print(f"  Exact Match: {exact_match:.4f}")
    return {"repo_id": model_name, "exact_match": exact_match, "num_samples": len(per_sample_exact),
            "per_sample_exact": per_sample_exact}

# ── HumanEval ──
def extract_code(generated: str, problem_prompt: str, entry_point: str) -> str:
    text = generated.strip()
    fence = re.search(r"```(?:python)?\s*\n?(.*?)```", text, re.DOTALL)
    if fence:
        text = fence.group(1).strip()
    if f"def {entry_point}" in text:
        return text
    return problem_prompt + "\n" + text

def _unsafe_execute(program: str, result_list, timeout: int):
    import signal
    def handler(signum, frame):
        raise TimeoutError("execution timed out")
    try:
        signal.signal(signal.SIGALRM, handler)
        signal.alarm(timeout)
        exec_globals = {}
        with contextlib.redirect_stdout(io.StringIO()):
            exec(program, exec_globals)
        signal.alarm(0)
        result_list.append("passed")
    except Exception as e:
        result_list.append(f"failed: {type(e).__name__}: {e}")

def check_correctness(problem: dict, completion_code: str, timeout: int = 5) -> bool:
    program = completion_code + "\n" + problem["test"] + f"\ncheck({problem['entry_point']})\n"
    manager = multiprocessing.Manager()
    result_list = manager.list()
    p = multiprocessing.Process(target=_unsafe_execute, args=(program, result_list, timeout))
    p.start()
    p.join(timeout=timeout + 1)
    if p.is_alive():
        p.kill(); p.join()
    if not result_list:
        result_list.append("failed: timeout")
    return result_list[0] == "passed"

def format_humaneval_prompt(problem_prompt: str) -> str:
    return ("Complete the following Python function. Return ONLY the complete "
            "function code (including the signature), with no explanations and "
            f"no markdown formatting.\n\n{problem_prompt}")

def evaluate_humaneval(model, tokenizer, model_name="model", num_samples=164, batch_size=4,
                        max_new_tokens=384, device="cuda"):
    print(f"\n{'─'*60}\n[HumanEval] Evaluating: {model_name}\n{'─'*60}")
    model.eval()
    prep_tokenizer_for_generation(tokenizer)
    dataset = load_dataset("openai/openai_humaneval", split="test")
    dataset = dataset.select(range(min(num_samples, len(dataset))))
    per_sample_pass = []
    for i in tqdm(range(0, len(dataset), batch_size), desc=f"{model_name} [humaneval]"):
        batch = dataset[i : i + batch_size]
        problem_prompts = batch["prompt"]
        entry_points    = batch["entry_point"]
        prompts = [build_chat_prompt(tokenizer, format_humaneval_prompt(p)) for p in problem_prompts]
        inputs = tokenizer(prompts, return_tensors="pt", padding=True, truncation=True,
                            max_length=768, add_special_tokens=False).to(device)
        with torch.no_grad():
            outputs = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False,
                                      pad_token_id=tokenizer.pad_token_id, eos_token_id=tokenizer.eos_token_id,
                                      repetition_penalty=1.1)
        input_len = inputs["input_ids"].shape[1]
        for j, output in enumerate(outputs):
            generated = tokenizer.decode(output[input_len:], skip_special_tokens=True)
            code = extract_code(generated, problem_prompts[j], entry_points[j])
            problem = {"prompt": problem_prompts[j], "test": batch["test"][j], "entry_point": entry_points[j]}
            per_sample_pass.append(int(check_correctness(problem, code, timeout=5)))
    pass_at_1 = round(sum(per_sample_pass) / len(per_sample_pass), 4)
    print(f"  pass@1: {pass_at_1:.4f}")
    return {"repo_id": model_name, "pass_at_1": pass_at_1, "num_samples": len(per_sample_pass),
            "per_sample_pass": per_sample_pass}

# ── Dolly-15k perplexity ──
def format_dolly_prompt(instruction: str, context: str) -> str:
    if context:
        return f"Instruction: {instruction}\nContext: {context}\nResponse:"
    return f"Instruction: {instruction}\nResponse:"

def evaluate_dolly_perplexity(model, tokenizer, model_name="model", num_samples=200,
                               max_length=512, device="cuda"):
    print(f"\n{'─'*60}\n[Dolly-PPL] Evaluating: {model_name}\n{'─'*60}")
    model.eval()
    dataset = load_dataset("databricks/databricks-dolly-15k", split="train")
    dataset = dataset.shuffle(seed=42).select(range(min(num_samples, len(dataset))))
    per_sample_nll, per_sample_ppl = [], []
    for ex in tqdm(dataset, desc=f"{model_name} [dolly-ppl]"):
        prompt = format_dolly_prompt(ex["instruction"], ex.get("context", ""))
        response = ex["response"]
        if not response.strip():
            continue
        full_text = prompt + " " + response
        prompt_ids = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=max_length).input_ids.to(device)
        full_ids   = tokenizer(full_text, return_tensors="pt", truncation=True, max_length=max_length).input_ids.to(device)
        if full_ids.shape[1] <= prompt_ids.shape[1]:
            continue
        labels = full_ids.clone()
        labels[:, : prompt_ids.shape[1]] = -100
        with torch.no_grad():
            out = model(full_ids, labels=labels)
        nll = out.loss.item()
        per_sample_nll.append(nll)
        per_sample_ppl.append(math.exp(nll))
    result = {"repo_id": model_name, "perplexity": round(sum(per_sample_ppl) / len(per_sample_ppl), 4),
              "mean_nll": round(sum(per_sample_nll) / len(per_sample_nll), 4),
              "num_samples": len(per_sample_ppl), "per_sample_nll": per_sample_nll}
    print(f"  Perplexity: {result['perplexity']:.4f}  (mean NLL: {result['mean_nll']:.4f})")
    return result


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Unsloth: `bitsandbytes` is not installed - 4bit QLoRA unallowed, but 16bit and full finetuning works!


/usr/local/lib/python3.12/dist-packages/unsloth/_gpu_init.py:390: UserWarning: Unsloth: Running `ldconfig /usr/lib64-nvidia` to link CUDA.
  warnings.warn("Unsloth: Running `ldconfig /usr/lib64-nvidia` to link CUDA.")
/sbin/ldconfig.real: /usr/local/lib/libtbbbind_2_5.so.3 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libur_adapter_level_zero.so.0 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libtbbbind.so.3 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libur_adapter_level_zero_v2.so.0 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libtbbbind_2_0.so.3 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libur_loader.so.0 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libtbb.so.12 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libumf.so.1 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libtcm_debug.so.1 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libur_adapter_opencl.so.0 

🦥 Unsloth Zoo will now patch everything to make training faster!


## Sanity Check: Confirm the Checkpoint Loads and Is Actually the Fixed Version

Load first, print a config field or two, BEFORE running the full eval. This is
the exact class of failure that derailed slerp-fixed last time (stale-checkpoint
references) — catch it here in seconds rather than after a 20+ minute eval run.

In [7]:
!pip install bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 MB 43.5 MB/s eta 0:00:00:00:0100:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
unsloth 2026.8.19 requires structlog>=24.1.0, which is not installed.
unsloth 2026.8.19 requires xformers>=0.0.27.post2; ("linux" in sys_platform or sys_platform == "win32") and (platform_machine == "AMD64" or platform_machine == "x86_64"), which is not installed.


In [8]:
SLERP_FIXED_REPO = "Srishtik/Qwen3-0.6B-slerp-FIXED-3-adapters-merged-2"

import torch
from unsloth import FastLanguageModel

_check_model, _check_tok = FastLanguageModel.from_pretrained(
    model_name     = SLERP_FIXED_REPO,
    max_seq_length = 1024,
    load_in_4bit   = True,
    dtype          = torch.float16,
)
print(f"Loaded OK: {SLERP_FIXED_REPO}")
print(f"Config dtype: {_check_model.config.torch_dtype}")
print(f"Num layers  : {_check_model.config.num_hidden_layers}")

del _check_model, _check_tok
torch.cuda.empty_cache()
print("\nSanity check passed — proceeding to full eval on this exact repo.")


Unsloth: `bitsandbytes` is unavailable here - disabling 4bit/8bit. 16bit LoRA and full finetuning still work.
==((====))==  Unsloth 2026.8.19: Fast Qwen3 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

`torch_dtype` is deprecated! Use `dtype` instead!


Loaded OK: Srishtik/Qwen3-0.6B-slerp-FIXED-3-adapters-merged-2
Config dtype: torch.float16
Num layers  : 28

Sanity check passed — proceeding to full eval on this exact repo.


## Run Full Eval

In [9]:
eval_model, eval_tokenizer = FastLanguageModel.from_pretrained(
    model_name     = SLERP_FIXED_REPO,
    max_seq_length = 1024,
    load_in_4bit   = True,
    dtype          = torch.float16,
)
FastLanguageModel.for_inference(eval_model)

slerp_fixed_gsm8k     = evaluate_gsm8k(eval_model, eval_tokenizer, model_name=SLERP_FIXED_REPO,
                                        num_samples=200, batch_size=4)
slerp_fixed_humaneval = evaluate_humaneval(eval_model, eval_tokenizer, model_name=SLERP_FIXED_REPO,
                                            num_samples=164, batch_size=4)
slerp_fixed_dolly     = evaluate_dolly_perplexity(eval_model, eval_tokenizer, model_name=SLERP_FIXED_REPO,
                                                   num_samples=200)

print(f"\n{'='*60}")
print("slerp-fixed summary:")
print(f"  GSM8K Exact Match : {slerp_fixed_gsm8k['exact_match']:.4f}")
print(f"  HumanEval pass@1  : {slerp_fixed_humaneval['pass_at_1']:.4f}")
print(f"  Dolly Perplexity  : {slerp_fixed_dolly['perplexity']:.4f}")
print(f"{'='*60}")

import gc, torch
del eval_model, eval_tokenizer
gc.collect()
torch.cuda.empty_cache()


Unsloth: `bitsandbytes` is unavailable here - disabling 4bit/8bit. 16bit LoRA and full finetuning still work.
==((====))==  Unsloth 2026.8.19: Fast Qwen3 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]


────────────────────────────────────────────────────────────
[GSM8K] Evaluating: Srishtik/Qwen3-0.6B-slerp-FIXED-3-adapters-merged-2
────────────────────────────────────────────────────────────


README.md: 0.00B [00:00, ?B/s]

main/train-00000-of-00001.parquet:   0%|          | 0.00/2.31M [00:00<?, ?B/s]

...

README.md: 0.00B [00:00, ?B/s]

databricks-dolly-15k.jsonl:   0%|          | 0.00/13.1M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/15011 [00:00<?, ? examples/s]

Srishtik/Qwen3-0.6B-slerp-FIXED-3-adapters-merged-2 [dolly-ppl]:   0%|          | 0/200 [00:00<?, ?it/s]`use_return_dict` is deprecated! Use `return_dict` instead!
Srishtik/Qwen3-0.6B-slerp-FIXED-3-adapters-merged-2 [dolly-ppl]: 100%|██████████| 200/200 [00:16<00:00, 11.84it/s]


  Perplexity: 11.9834  (mean NLL: 1.8951)

slerp-fixed summary:
  GSM8K Exact Match : 0.0150
  HumanEval pass@1  : 0.1463
  Dolly Perplexity  : 11.9834


## Final Comparison — All 8 Methods + 3 Specialists

In [10]:
known_results = {
  "linear":                          {"gsm8k": 0.1100, "humaneval": 0.2012, "dolly_ppl": 17.8096},
  "svd":                             {"gsm8k": 0.1550, "humaneval": 0.2073, "dolly_ppl": 20.5070},
  "ties":                            {"gsm8k": 0.0550, "humaneval": 0.2073, "dolly_ppl": 16.7428},
  "dare":                            {"gsm8k": 0.0400, "humaneval": 0.2378, "dolly_ppl": 17.2781},
  "bwsum":                           {"gsm8k": 0.1200, "humaneval": 0.1829, "dolly_ppl": 18.8800},
  "slerp (buggy, best order)":       {"gsm8k": 0.0100, "humaneval": 0.2073, "dolly_ppl": 13.1499},
  "ct-calibrated":                   {"gsm8k": 0.0600, "humaneval": 0.1951, "dolly_ppl": 12.9480},
  "codealpaca_adapter (specialist)": {"gsm8k": 0.0850, "humaneval": 0.1707, "dolly_ppl": 38.3633},
  "metamath_adapter (specialist)":   {"gsm8k": 0.2200, "humaneval": 0.1220, "dolly_ppl": 28.6676},
  "dolly_adapter (specialist)":      {"gsm8k": 0.0250, "humaneval": 0.1646, "dolly_ppl": 12.6278},
}

print(f"{'Model':<40}{'GSM8K':>10}{'HumanEval':>12}{'Dolly PPL':>12}")
print("-" * 74)
for name, r in known_results.items():
    print(f"{name:<40}{r['gsm8k']:>10.4f}{r['humaneval']:>12.4f}{r['dolly_ppl']:>12.4f}")
print(f"{'slerp-fixed (NEW, closes final open item)':<40}"
      f"{slerp_fixed_gsm8k['exact_match']:>10.4f}"
      f"{slerp_fixed_humaneval['pass_at_1']:>12.4f}"
      f"{slerp_fixed_dolly['perplexity']:>12.4f}")
print("=" * 74)

best_gsm8k = max(known_results.items(), key=lambda kv: kv[1]["gsm8k"])
best_he    = max(known_results.items(), key=lambda kv: kv[1]["humaneval"])
best_ppl   = min(known_results.items(), key=lambda kv: kv[1]["dolly_ppl"])

print("\nRanking check vs. all 7 prior merge methods:")
print(f"  GSM8K     — best prior: {best_gsm8k[0]} ({best_gsm8k[1]['gsm8k']:.4f})  "
      f"vs slerp-fixed: {slerp_fixed_gsm8k['exact_match']:.4f}")
print(f"  HumanEval — best prior: {best_he[0]} ({best_he[1]['humaneval']:.4f})  "
      f"vs slerp-fixed: {slerp_fixed_humaneval['pass_at_1']:.4f}")
print(f"  Dolly PPL — best prior: {best_ppl[0]} ({best_ppl[1]['dolly_ppl']:.4f})  "
      f"vs slerp-fixed: {slerp_fixed_dolly['perplexity']:.4f}")


Model                                        GSM8K   HumanEval   Dolly PPL
--------------------------------------------------------------------------
linear                                      0.1100      0.2012     17.8096
svd                                         0.1550      0.2073     20.5070
ties                                        0.0550      0.2073     16.7428
dare                                        0.0400      0.2378     17.2781
bwsum                                       0.1200      0.1829     18.8800
slerp (buggy, best order)                   0.0100      0.2073     13.1499
ct-calibrated                               0.0600      0.1951     12.9480
codealpaca_adapter (specialist)             0.0850      0.1707     38.3633
metamath_adapter (specialist)               0.2200      0.1220     28.6676
dolly_adapter (specialist)                  0.0250      0.1646     12.6278
slerp-fixed (NEW, closes final open item)    0.0150      0.1463     11.9834

Ranking check vs. all 7